# 17 · Area around a point, against theory

**Question.** Does Fieldwork's `buffer_point` widget draw the circle it claims to, and how good
an approximation is it?

**This is the first widget in this lab that cannot agree exactly with a reference, and this
check says so.** Turf buffers in a local azimuthal equidistant projection, so there is no
second implementation of the same construction to compare against. `buffer_area` is the same
construction at a coarser setting and has its own check, 18.

What *can* be checked exactly is the geometry of the approximation. A circle buffered with
`steps` segments per quadrant is a polygon of `4 × steps` vertices inscribed in it, and the
area of such a polygon has a closed form:

$$\frac{n}{2\pi}\sin\frac{2\pi}{n}$$

That is 0.99359 at 32 vertices and 0.99960 at 128 — exactly the figures Fieldwork's own
comments record from measurement. So the widget's reported `circleAreaFraction` can be checked
against **theory**, not against another implementation of the same thing.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_17_buffer_point import run

report = run(Path.cwd() / "fixtures" / "buffer-point-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print()
print(f"  steps per quadrant      {report['stepsPerQuadrant']}")
print(f"  vertices                {report['vertexCount']} (expected {report['expectedVertexCount']})")
print(f"  circle-area fraction    {report['reportedCircleAreaFraction']} "
      f"against closed form {report['closedFormFraction']:.5f}")
print(f"  worst radius error      {report['worstVertexRadiusErrorMetres']:.3e} m on the sphere")
print()
print(f"  on the sphere  {report['sphericalRadiusRangeMetres'][0]:.4f} to "
      f"{report['sphericalRadiusRangeMetres'][1]:.4f} m")
print(f"  on WGS84       {report['ellipsoidalRadiusRangeMetres'][0]:.4f} to "
      f"{report['ellipsoidalRadiusRangeMetres'][1]:.4f} m")

## The criterion that was wrong, and what it found

**This check failed on its first run, and the failure was in the criterion rather than in the
widget.** It asked that every vertex be within 1 m of the stated 500 m radius, measured as a
geodesic distance on the WGS84 ellipsoid. The worst was 1.589 m.

Investigating instead of widening the tolerance produced the most useful fact in this
notebook: Turf buffers on a **sphere of radius 6,371,008.8 m**, and does so exactly.

In [ ]:
import pyproj

fixture = json.loads((Path.cwd() / "fixtures" / "buffer-point-soho-001.json").read_text())
centre = fixture["input"]["point"]["coordinates"]
ring = fixture["output"]["boundary"]["coordinates"][0][:-1]
radius = fixture["declaredParameters"]["radiusMetres"]

sphere = pyproj.Geod(a=6371008.8, b=6371008.8)
wgs84 = pyproj.Geod(ellps="WGS84")

for label, model in [("sphere 6371008.8 m", sphere), ("WGS84 ellipsoid   ", wgs84)]:
    distances = [model.inv(centre[0], centre[1], x, y)[2] for x, y in ring]
    print(f"{label}  min {min(distances):9.4f}  max {max(distances):9.4f}  "
          f"spread {max(distances) - min(distances):7.4f} m")

print()
print(f"The widget says {radius} m. On the sphere it draws {radius} m, to four decimals, in every")
print("direction. On the ellipsoid the same ring is up to 1.59 m further out, and by a different")
print("amount in each direction -- which is why the spread is not zero there.")
print()
print(f"As a fraction of the radius that is {1.589 / radius:.2%}. On a 5 km scope it would be")
print(f"about {1.589 / radius * 5000:.0f} m.")

### Why this matters more than the tolerance did

A practitioner reading "everything within 500 m of this point" will take it to mean 500 m on
the ground. It is 500 m on a sphere, and **500.28 to 501.59 m on the ground**, varying with
direction. Nothing is wrong with the widget: a spherical buffer is a reasonable construction
and the error is a third of a percent. But it was not written down anywhere before this check
was attempted, and it was found only because a criterion fixed in advance failed and was
investigated rather than relaxed.

That is the argument for fixing criteria before running. A tolerance chosen after seeing the
answer would have been 2 m, would have passed, and would have found nothing.

## Dive deeper · what the step count buys

`buffer_point` uses 32 steps where `buffer_area` uses 8, and the widget's comment explains
why: a scope's edge decides membership, so the finer setting is worth it there. The closed
form says exactly what each setting costs.

In [ ]:
import math

def fraction(vertices):
    return (vertices / (2 * math.pi)) * math.sin(2 * math.pi / vertices)

print(f"{'steps/quadrant':>14} {'vertices':>9} {'area of the circle':>20} {'missing':>10}")
for steps in (1, 2, 4, 8, 16, 32, 64):
    vertices = 4 * steps
    share = fraction(vertices)
    mark = "  <- buffer_area" if steps == 8 else ("  <- buffer_point" if steps == 32 else "")
    print(f"{steps:>14} {vertices:>9} {share:>19.5%} {1 - share:>9.3%}{mark}")

print()
radius = fixture["declaredParameters"]["radiusMetres"]
ideal = math.pi * radius ** 2
print(f"On a {radius} m radius, {ideal:.0f} m2 of ideal circle:")
for steps in (8, 32):
    missing = ideal * (1 - fraction(4 * steps))
    print(f"  {steps:>2} steps leaves {missing:7.1f} m2 outside the polygon "
          f"({missing / ideal:.3%})")
print()
print("Going from 8 to 32 recovers most of that, and doubling again recovers little. The")
print("widget's choice is at the knee of the curve, which is the useful thing to see.")

### Reading the result

`agrees` means the vertex count and the area fraction match theory exactly, and the circle is
the stated radius on the sphere it is drawn on.

It does not mean a 500 m buffer is 500 m on the ground — it is up to 1.59 m more — and it
does not mean the radius is the right radius. A buffered scope is a decision, not a surveyed
boundary: its edge decides membership and its radius is an assumption of the question.